# Apuntes de Teoría – Sesiones 23 al 27
## ISIS-2611 | Aprendizaje de Máquina

---

| Sesión | Tema |
|--------|------|
| S23 | Entrenamiento de Redes Neuronales (Backpropagation) |
| S24 | CNN y Transfer Learning |
| S25 | Modelos Secuenciales (RNN, LSTM, GRU) |
| S26 | Semántica Vectorial (Embeddings, Word2Vec, BERT) |
| S27 | Dilemas Éticos en IA |


---
---
# SESIÓN 23 – Entrenamiento de Redes Neuronales
---

## 1. ¿Por qué necesitamos redes multicapa?

Un perceptrón simple (una sola neurona) solo puede separar clases **linealmente separables**.
Si los datos no se pueden separar con una línea recta, necesitamos capas ocultas.

Cada neurona oculta aprende un **límite de decisión** parcial.
La capa de salida combina esos límites para formar fronteras no lineales complejas.

```
Entrada x → [capa oculta: z1, z2...] → [salida: ŷ]

z1 = f(x1·w11 + x2·w12 + b1)   ← límite de decisión 1
z2 = f(x1·w21 + x2·w22 + b2)   ← límite de decisión 2
ŷ  = f(z1·w31 + z2·w32 + b3)   ← combina z1 y z2
```

La función `f` es la **función de activación** (ej: sigmoid, relu).
Sin ella, apilar capas seguiría siendo una transformación lineal.

## 2. Función de pérdida J

Mide qué tan equivocado está el modelo en todos los ejemplos de entrenamiento.

**MSE (para regresión):**
$$J = \sum_k \frac{1}{2}(y_k - \hat{y}_k)^2$$

El objetivo del entrenamiento es **minimizar J** ajustando los pesos.

Para minimizar J usamos **descenso de gradiente**:
$$\omega_{nuevo} = \omega_{viejo} - \eta \frac{\partial J}{\partial \omega_{viejo}}$$

Donde:
- $\eta$ = **learning rate** (tasa de aprendizaje): controla el tamaño del paso
- $\frac{\partial J}{\partial \omega}$ = gradiente: dirección de máximo crecimiento de J
- El **signo negativo** hace que el peso se mueva en la dirección de mínimo J

El **sesgo** (bias) se actualiza igual:
$$b_{nuevo} = b_{viejo} - \eta \cdot \delta$$

## 3. Retropropagación (Backpropagation)

Es el algoritmo que calcula **cómo afecta cada peso a la pérdida J**.
Usa la **regla de la cadena** del cálculo para propagar el error desde la salida hacia la entrada.

### Ejemplo con 2 neuronas en serie: x → N1 → N2 → ŷ

**Paso 1 – Forward pass:** calcular ŷ con los pesos actuales.

**Paso 2 – Calcular gradiente de la capa de salida (N2):**

$$\frac{\partial J}{\partial \omega_{11}^{(2)}} = \underbrace{\frac{\partial J}{\partial \hat{y}}}_{-(y-\hat{y})} \cdot \underbrace{\frac{\partial \hat{y}}{\partial net_2}}_{\hat{y}(1-\hat{y})} \cdot \underbrace{\frac{\partial net_2}{\partial \omega_{11}^{(2)}}}_{y^{(1)}}$$

Resultado:
$$\frac{\partial J}{\partial \omega_{11}^{(2)}} = -(y - \hat{y}) \cdot \hat{y}(1-\hat{y}) \cdot y^{(1)} = \delta^{(2)} \cdot y^{(1)}$$

**Paso 3 – Calcular gradiente de la capa oculta (N1):**

$$\frac{\partial J}{\partial \omega_{11}^{(1)}} = -(y - \hat{y}) \cdot \hat{y}(1-\hat{y}) \cdot \omega_{11}^{(2)} \cdot y^{(1)}(1-y^{(1)}) \cdot x_1$$

El error se "propaga hacia atrás" a través de los pesos $\omega_{11}^{(2)}$.

### El delta (δ)
El delta es el **error local** de cada neurona. Se reutiliza en capas anteriores:
- Capa de salida: $\delta^{(2)} = -(y - \hat{y}) \cdot \hat{y}(1-\hat{y})$
- Capa oculta: $\delta^{(1)} = \delta^{(2)} \cdot \omega^{(2)} \cdot y^{(1)}(1-y^{(1)})$

### Derivada de la sigmoid (necesaria en backprop)
$$\frac{d\sigma(x)}{dx} = \sigma(x)(1 - \sigma(x))$$

Esta derivada aparece en TODOS los gradientes cuando la activación es sigmoid.

## 4. Problemas del entrenamiento y cómo resolverlos

### Vanishing Gradient (gradiente que desaparece)
- **Qué es:** en redes muy profundas con sigmoid, la derivada $\sigma(1-\sigma) \leq 0.25$.
  Al multiplicar muchas veces este valor, el gradiente se hace casi 0 en las primeras capas.
- **Consecuencia:** las primeras capas dejan de aprender.
- **Solución:** usar activación **ReLU** en las capas ocultas (gradiente = 1 si x > 0).

### Exploding Gradient (gradiente que explota)
- **Qué es:** el gradiente crece descontroladamente → pesos = NaN.
- **Causas:** learning rate muy alto, arquitectura muy profunda sin normalización.
- **Solución:** reducir lr, usar **Gradient Clipping**, BatchNormalization.

### Overfitting
- El modelo memoriza el train set, no generaliza al test.
- **Señal:** train_loss baja, val_loss sube (curvas divergentes).
- **Soluciones:**
  - `Dropout(p)`: desactiva aleatoriamente p% de neuronas en cada paso.
  - Regularización L1/L2: penaliza pesos grandes.
  - EarlyStopping: detiene el entrenamiento cuando val_loss no mejora.
  - Más datos (o data augmentation).

### Learning Rate
| Valor | Efecto |
|-------|--------|
| Muy alto (>0.1) | Loss oscila o explota a NaN |
| Adecuado (~0.001) | Converge estable |
| Muy bajo (<0.00001) | Converge muy despacio |

### Batch Size y Épocas
- **Epoch:** una pasada completa por todos los datos de entrenamiento.
- **Batch size:** cuántos ejemplos se procesan antes de actualizar los pesos.
  - Batch grande → estimación estable del gradiente, más RAM.
  - Batch pequeño (ej: 32) → más ruido pero puede escapar de mínimos locales.
- **Stochastic GD:** batch size = 1.
- **Mini-batch GD:** batch size entre 32 y 512 (lo más común).

## 5. Optimizadores

Variantes del descenso de gradiente que convergen más rápido o mejor.

| Optimizador | Característica | Cuándo usarlo |
|-------------|---------------|--------------|
| SGD | Básico, sin memoria | Problemas simples |
| **Adam** | Combina momentum + RMSProp, adapta lr por parámetro | **Default en la mayoría de casos** |
| RMSProp | Adapta lr según la magnitud reciente del gradiente | RNNs, series de tiempo |
| Adagrad | Reduce lr para params frecuentes | NLP, datos dispersos |

**Adam** (Adaptive Moment Estimation) es el default recomendado. lr = 0.001.

## 6. BatchNormalization

Normaliza las activaciones de cada capa para que tengan media ≈ 0 y std ≈ 1.

**Beneficios:**
- Estabiliza el entrenamiento (reduce el problema de "Internal Covariate Shift").
- Permite usar learning rates más altos.
- Actúa como regularizador leve.

**Posición correcta:** después de `Dense` o `Conv2D`, antes de la activación.
```python
layers.Dense(256)
layers.BatchNormalization()   # ← aquí
layers.Activation('relu')
layers.Dropout(0.3)
```
**Nunca** poner BN justo antes de la capa de salida.

---
---
# SESIÓN 24 – CNN y Transfer Learning
---

## 1. ¿Por qué CNN para imágenes?

Un MLP con imágenes aplana los píxeles y pierde la información **espacial** (qué está al lado de qué).
Una CNN preserva esa información usando **filtros** que recorren la imagen.

**Ventajas de CNN:**
- **Compartición de pesos:** el mismo filtro se aplica en toda la imagen → menos parámetros.
- **Invarianza traslacional:** reconoce el mismo objeto sin importar dónde esté en la imagen.
- **Jerarquía de features:** capas iniciales detectan bordes, capas medias formas, capas finales objetos.

## 2. Operación de Convolución

Un filtro (kernel) de tamaño k×k se desliza sobre la imagen multiplicando elemento a elemento y sumando.

```
Imagen (H×W×C)  →  Conv2D (filtros=32, kernel=3×3)  →  Feature Maps (H'×W'×32)
```

- **Filtros:** cada filtro aprende a detectar un tipo de patrón diferente.
- **Kernel size:** 3×3 es el más común. Más grande captura más contexto pero más parámetros.
- **Stride:** cuánto se desplaza el filtro en cada paso.
  - stride=1: máximo detalle.
  - stride=2: reduce el tamaño a la mitad (como un pooling).

### Padding
| Tipo | Qué hace | Efecto en tamaño |
|------|----------|------------------|
| `valid` (default) | Sin relleno | Reduce H y W: out = (in - k + 1) |
| `same` | Rellena con ceros | Mantiene H y W igual |

**Problema con `valid` en imágenes pequeñas:** después de varias capas Conv, el tamaño llega a 0 → usar `padding='same'`.

## 3. Pooling

Reduce el tamaño espacial de los feature maps manteniendo la información más importante.

| Tipo | Qué hace |
|------|----------|
| `MaxPooling2D(2,2)` | Toma el valor máximo de cada ventana 2×2 → reduce a la mitad |
| `GlobalAveragePooling2D` | Promedia cada feature map entero → produce un vector 1D sin depender del tamaño |

**GlobalAveragePooling2D** es útil en transfer learning porque el modelo acepta cualquier tamaño de imagen.

## 4. Arquitectura CNN completa

```
INPUT (32×32×3)
  │
  ▼
Conv2D(32, 3×3, padding='same', relu)   → (32×32×32)   detecta: bordes
MaxPooling2D(2×2)                       → (16×16×32)   reduce
Dropout(0.2)
  │
  ▼
Conv2D(64, 3×3, padding='same', relu)   → (16×16×64)   detecta: formas
MaxPooling2D(2×2)                       → (8×8×64)
Dropout(0.2)
  │
  ▼
FLATTEN()                               → (4096,)       ← OBLIGATORIO
Dense(128, relu)
Dense(10, softmax)                      → predicción de 10 clases
```

### Reglas críticas
- `Flatten()` es **obligatorio** entre la parte convolucional y `Dense`.
- Imágenes deben normalizarse: `imagen / 255.0` → rango [0,1].
- `input_shape` siempre incluye el canal: `(H, W, C)` → ej: `(28,28,1)` o `(32,32,3)`.
- Para escala de grises: reshape necesario → `X.reshape(-1, H, W, 1)`.

## 5. Transfer Learning

Reutilizar un modelo ya entrenado en un dataset grande (ImageNet, ~1.2M imágenes, 1000 clases)
para resolver un problema nuevo con menos datos.

**Idea:** las primeras capas de una CNN aprenden features generales (bordes, texturas, formas)
que son útiles para cualquier problema de visión.

### Modelos pre-entrenados disponibles en Keras
| Modelo | Característica |
|--------|----------------|
| VGG16 / VGG19 | Simple, muy conocido, pesado |
| ResNet50 | Conexiones residuales, muy profundo |
| **MobileNetV2** | Ligero, rápido, bueno para móviles |
| EfficientNet | Estado del arte en eficiencia |

### Fase 1: Feature Extraction (solo entrenar el "top")
```python
base = tf.keras.applications.MobileNetV2(input_shape=(224,224,3),
                                          include_top=False,
                                          weights='imagenet')
base.trainable = False   # ← CONGELAR: no modificar los pesos de ImageNet

model = keras.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(N_clases, activation='softmax')
])
model.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy')
```

### Fase 2: Fine-tuning (opcional, descongelar capas superiores)
```python
base.trainable = True
# Recompilar SIEMPRE después de cambiar trainable
model.compile(optimizer=Adam(0.00001),  # lr MUY bajo para no destruir los pesos
              loss='sparse_categorical_crossentropy')
```

### Preprocesamiento específico de cada modelo
Cada arquitectura fue entrenada con su propio proceso. Usar `preprocess_input` del módulo correcto:
```python
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
X_prep = preprocess_input(X.astype('float32'))  # ← NO dividir entre 255 a mano
```

### ¿Cuándo congelar vs entrenar?
| Datos disponibles | Estrategia |
|-------------------|------------|
| Pocos datos, similar a ImageNet | Solo entrenar el top (congelar base) |
| Pocos datos, muy diferente a ImageNet | Fine-tuning con lr bajo |
| Muchos datos | Entrenar todo desde cero o fine-tuning agresivo |

### Data Augmentation
Técnica para ampliar artificialmente el conjunto de entrenamiento.
- Rotaciones, flips, zoom, desplazamientos → el modelo ve más variedad.
- **Solo se aplica al train set, NUNCA al test set.**
```python
train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=40, horizontal_flip=True)
test_datagen  = ImageDataGenerator(rescale=1./255)   # ← solo normalizar
```

---
---
# SESIÓN 25 – Modelos Secuenciales (RNN, LSTM, GRU)
---

## 1. ¿Por qué modelos secuenciales?

Un MLP procesa cada ejemplo **de forma independiente** — ignora el orden.
Para datos donde el orden importa (texto, audio, series de tiempo), necesitamos modelos que tengan **memoria**.

Ejemplos:
- "El banco aprobó el préstamo" — el orden de las palabras cambia el significado.
- La temperatura de hoy depende de la temperatura de ayer.

## 2. RNN – Red Neuronal Recurrente

En cada paso de tiempo t, la red recibe:
- La entrada actual: $x_t$
- El estado oculto del paso anterior: $h_{t-1}$

Y produce:
- El nuevo estado oculto: $h_t = f(W_h \cdot h_{t-1} + W_x \cdot x_t + b)$

```
x1 → [RNN] → h1
       ↑
       h0
x2 → [RNN] → h2
       ↑
       h1
...etc
```

### Problema del SimpleRNN: Vanishing Gradient

Al hacer backpropagation a través del tiempo (BPTT), el gradiente se multiplica repetidamente por valores pequeños.
En secuencias largas (>30 pasos), el gradiente llega casi en 0 a los primeros pasos → la red **olvida** el inicio.

**Consecuencia práctica:** SimpleRNN no puede capturar dependencias a largo plazo.

## 3. LSTM – Long Short-Term Memory

Soluciona el vanishing gradient con un **estado de celda** (C) que fluye por la red
con pocas modificaciones, permitiendo que la información persista muchos pasos.

### Las 3 compuertas del LSTM

| Compuerta | Función | Activación |
|-----------|---------|------------|
| **Forget gate** (f) | Decide qué olvidar del estado anterior | sigmoid → [0,1] |
| **Input gate** (i) | Decide qué nueva información agregar | sigmoid → [0,1] |
| **Output gate** (o) | Decide qué parte del estado producir como salida | sigmoid → [0,1] |

```
Estado de celda:  Ct = f·Ct-1 + i·C̃t
Estado oculto:    ht = o · tanh(Ct)
```

- Si f ≈ 1 → recuerda todo el pasado.
- Si f ≈ 0 → olvida el pasado (útil cuando hay un tema nuevo).

### return_sequences — la regla más importante

```
return_sequences=False (default):
  Salida: (batch, units) ← solo el último estado
  Usar cuando: el siguiente es Dense o hay un solo LSTM

return_sequences=True:
  Salida: (batch, timesteps, units) ← un estado por cada paso
  Usar cuando: el siguiente es OTRO LSTM/GRU
```

**LSTM apilados:**
```python
layers.LSTM(128, return_sequences=True),   # ← intermedio
layers.LSTM(64,  return_sequences=True),   # ← intermedio
layers.LSTM(32,  return_sequences=False),  # ← último → va a Dense
layers.Dense(1)
```

## 4. GRU – Gated Recurrent Unit

Versión simplificada del LSTM. Tiene solo **2 compuertas** (reset y update) en vez de 3.

| | LSTM | GRU |
|-|------|-----|
| Compuertas | 3 (forget, input, output) | 2 (reset, update) |
| Parámetros | Más | Menos |
| Velocidad | Más lento | Más rápido |
| Rendimiento | Similar | Similar |

**Cuándo usar GRU:** cuando el tiempo de entrenamiento importa y el dataset es grande.
En la práctica, LSTM y GRU dan resultados similares — se puede usar cualquiera.

## 5. Bidireccional

Procesa la secuencia en ambos sentidos: →  y  ←
Tiene acceso al contexto **futuro y pasado** de cada token.

```python
layers.Bidirectional(layers.LSTM(64))
# Salida: (batch, 128) — duplica las unidades porque hay dos direcciones
```

| Tarea | Bidireccional |
|-------|---------------|
| Clasificación de texto | ✅ sí — tienes toda la oración |
| Traducción (encoder) | ✅ sí |
| **Generación de texto** | ❌ no — ve el futuro, hace trampa |
| **Series de tiempo** | ❌ no — el futuro no existe |

## 6. Series de tiempo con LSTM

El LSTM espera input 3D: `(muestras, pasos_de_tiempo, features)`

```python
# Si tienes una sola variable (ej: precio)
X = X.reshape(X.shape[0], X.shape[1], 1)   # (N, 30) → (N, 30, 1)

model = keras.Sequential([
    layers.LSTM(64, input_shape=(30, 1)),
    layers.Dense(1)                   # regresión: sin activación
])
model.compile('adam', 'mse')
```

## 7. Padding en secuencias de texto

Para que todas las secuencias tengan la misma longitud, se añaden ceros.

| padding | Dónde van los ceros | Mejor para |
|---------|---------------------|------------|
| `post` | al final | CNN 1D |
| `pre` | al inicio | **LSTM** (el estado final refleja el texto real) |

Se puede usar `Masking(mask_value=0)` para que la red ignore los pasos de padding.

---
---
# SESIÓN 26 – Semántica Vectorial y Embeddings
---

## 1. El problema de representar texto

Las redes solo procesan números. ¿Cómo convertir palabras en números de manera útil?

### Opción 1: One-Hot Encoding
Cada palabra es un vector con un 1 en su posición y 0 en el resto.

```
Vocabulario: [gato, perro, banco, río]
"gato" → [1, 0, 0, 0]
"perro"→ [0, 1, 0, 0]
```

**Problemas:**
- Dimensión = tamaño del vocabulario (puede ser 50.000+) → inviable.
- Todos los vectores son **ortogonales** → "rey" y "reina" son igual de diferentes que "rey" y "autobús".
- No captura relaciones semánticas.

### Opción 2: Embedding
Cada palabra se mapea a un vector denso de dimensión fija (ej: 64 o 300).
Estos vectores se **aprenden** durante el entrenamiento.

```
"gato"  → [0.23, -0.51, 0.88, ..., 0.14]   (64 valores)
"felino"→ [0.21, -0.49, 0.91, ..., 0.16]   (similar al de gato)
"banco" → [0.78,  0.12, -0.34, ..., 0.45]  (muy diferente)
```

**Ventajas:**
- Compacto (64 vs 50.000 dimensiones).
- Palabras similares quedan **cerca** en el espacio vectorial.
- Captura analogías: rey − hombre + mujer ≈ reina.

## 2. Similitud coseno

Mide cuán similares son dos vectores sin importar su magnitud.

$$\text{similitud}(A, B) = \frac{A \cdot B}{\|A\| \cdot \|B\|}$$

- Resultado = 1 → vectores idénticos (misma dirección).
- Resultado = 0 → vectores ortogonales (sin relación).
- Resultado = -1 → vectores opuestos.

**Uso:** comparar similitud entre palabras, oraciones o documentos en el espacio de embeddings.

## 3. Word2Vec

Modelo que aprende embeddings de palabras a partir de su **contexto** en grandes corpus de texto.

**Hipótesis distribucional:** "Conocerás una palabra por la compañía que mantiene."
Palabras que aparecen en contextos similares tienen significados similares.

### Dos arquitecturas

| Variante | Tarea de entrenamiento |
|----------|------------------------|
| **CBOW** (Continuous Bag of Words) | Predecir la palabra central dado su contexto |
| **Skip-gram** | Predecir el contexto dado la palabra central |

El modelo aprende a maximizar la probabilidad del contexto → los pesos de la capa oculta **son** los embeddings.

### Propiedades
- Captura analogías semánticas y sintácticas.
- Limitación: una palabra tiene **un solo vector** sin importar el contexto.
  - "banco" (financiero) vs "banco" (río) → mismo embedding.

## 4. Embeddings contextuales: BERT

**BERT** (Bidirectional Encoder Representations from Transformers) genera embeddings que
**cambian según el contexto** de la oración.

```
"banco" en "fui al banco a depositar" → vector [0.3, -0.2, ...] (financiero)
"banco" en "me senté en el banco del parque" → vector [0.8, 0.5, ...] (mueble)
```

### Estructura de BERT
- Basado en **Transformers** (mecanismo de atención).
- `bert-base-uncased`: 12 capas, 768 dimensiones, 110M parámetros.
- `bert-base-multilingual-cased`: soporta 104 idiomas.

### Cómo usar BERT para clasificación

```
texto → tokenizador BERT → tokens → modelo BERT → embeddings (N_tokens × 768)
                                                          ↓
                                              token [CLS] (posición 0) ← representación de la oración
                                                          ↓
                                              clasificador (Dense / LogReg)
```

El **token [CLS]** es el token especial que BERT usa para codificar el significado global de la oración.

```python
outputs = modelo_bert(**inputs)
cls_embedding = outputs.last_hidden_state[:, 0, :]   # posición 0 = [CLS]
# shape: (batch, 768)
```

**Alternativa:** promedio de todos los tokens (mean pooling):
```python
token_embeddings = outputs.last_hidden_state[:, 1:-1, :]  # excluye [CLS] y [SEP]
mean_embedding = token_embeddings.mean(dim=1)
```

## 5. Reducción de dimensionalidad para visualizar

Los embeddings son vectores de alta dimensión (768, 300...) — no se pueden graficar directamente.

| Técnica | Característica |
|---------|----------------|
| **PCA** | Lineal, rápido, preserva varianza global |
| **t-SNE** | No lineal, preserva estructura local, lento |

```python
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(embeddings)   # de (N, 768) a (N, 2)
```

## 6. TF-IDF (alternativa sin redes)

**TF-IDF** = Term Frequency × Inverse Document Frequency

- TF: cuántas veces aparece la palabra en el documento.
- IDF: log(N_documentos / documentos_que_contienen_la_palabra) → penaliza palabras muy comunes ("el", "de").

Produce vectores dispersos pero eficientes para clasificación de texto con modelos clásicos (Naive Bayes, SVM).
**No captura semántica** (como Word2Vec sí lo hace).

---
---
# SESIÓN 27 – Dilemas Éticos en IA
---

## 1. ¿Por qué la ética en IA?

Los modelos de ML no son objetivos. Aprenden de datos producidos por humanos,
y esos datos contienen los sesgos de la sociedad.
Un modelo que reproduce esos sesgos puede causar daño real a escala masiva.

## 2. Tipos de sesgo (bias) en IA

| Tipo | Descripción | Ejemplo |
|------|-------------|--------|
| **Sesgo de representación** | El dataset no representa bien a todos los grupos | Modelo de reconocimiento facial entrenado solo con caras blancas |
| **Sesgo de medición** | La variable objetivo fue medida de forma distinta para diferentes grupos | Diagnóstico médico más frecuente en pacientes con seguro |
| **Sesgo de agregación** | Se usa un modelo único para grupos con comportamientos distintos | Un modelo de diabetes entrenado en hombres aplicado a mujeres |
| **Sesgo histórico** | El pasado refleja injusticias que el modelo perpetúa | Algoritmo de contratación que penaliza CVs de mujeres |
| **Sesgo de retroalimentación** | Las predicciones afectan los datos futuros | Más policías en zonas pobres → más arrestos → el modelo cree que son más peligrosas |

## 3. Dimensiones éticas clave

### Fairness (Equidad)
El modelo no debe discriminar injustamente a grupos protegidos (género, raza, edad, etc.).

**El problema de las métricas de equidad:**
Es matemáticamente imposible satisfacer todas las definiciones de equidad al mismo tiempo.

| Definición | Qué dice |
|-----------|----------|
| Paridad demográfica | La misma tasa de predicciones positivas en todos los grupos |
| Igualdad de oportunidad | La misma tasa de verdaderos positivos (recall) en todos los grupos |
| Calibración | La probabilidad predicha refleja la probabilidad real |

### Transparencia e Interpretabilidad
¿Podemos explicar por qué el modelo tomó una decisión?
- Modelos interpretables: regresión lineal, árboles de decisión.
- Modelos de caja negra: redes neuronales profundas.
- Técnicas de explicabilidad: **LIME**, **SHAP** (explican predicciones individuales).

### Privacidad
Los modelos entrenados en datos personales pueden **memorizar** ejemplos de entrenamiento.
Técnicas: differential privacy, federated learning.

### Accountability (Responsabilidad)
¿Quién es responsable cuando un modelo toma una decisión incorrecta y causa daño?

## 4. Casos reales de dilemas éticos

| Caso | Problema ético |
|------|----------------|
| COMPAS (justicia criminal EEUU) | Algoritmo de reincidencia más preciso para blancos que para negros |
| Reconocimiento facial | Tasas de error mucho más altas en mujeres de piel oscura |
| Amazon recruiting tool | Penalizaba CVs que incluían la palabra "mujeres" |
| Préstamos bancarios | Algoritmos niegan créditos en zonas históricamente redlineadas |
| Diagnóstico médico | Modelos entrenados en datos de países ricos fallan en pacientes de países pobres |

## 5. Principios de IA responsable

La mayoría de marcos regulatorios (UE AI Act, Google AI Principles, etc.) coinciden en:

1. **Beneficencia:** la IA debe beneficiar a las personas.
2. **No maleficencia:** minimizar daños y efectos adversos.
3. **Autonomía:** respetar la capacidad de decisión de las personas.
4. **Justicia:** distribución equitativa de beneficios y daños.
5. **Explicabilidad:** los afectados deben poder entender las decisiones.

## 6. Checklist ético para proyectos de ML

- [ ] ¿El dataset representa a todos los grupos relevantes?
- [ ] ¿Hay variables proxy que codifiquen atributos protegidos? (ej: código postal puede codificar raza)
- [ ] ¿Cuál es el costo de un falso positivo vs un falso negativo para distintos grupos?
- [ ] ¿Quién tiene acceso a los datos? ¿Son datos personales?
- [ ] ¿Es posible explicar las predicciones individuales?
- [ ] ¿Se monitorea el modelo en producción para detectar degradación o nuevos sesgos?
- [ ] ¿Existe un mecanismo para que los afectados puedan apelar una decisión automatizada?

---
---
# RESUMEN FINAL – Todo en una página
---

## S23 – Entrenamiento de RN
```
Descenso de gradiente:  ω_nuevo = ω_viejo − η · (∂J/∂ω)
Backprop:               propaga el error de salida a entrada usando regla de la cadena
δ:                      error local de cada neurona, se reutiliza en capas anteriores
Sigmoid:                dσ/dx = σ(x)(1−σ(x))  ← aparece en todos los gradientes
Problemas:
  - Vanishing gradient → usar ReLU en capas ocultas
  - Exploding gradient → reducir lr, gradient clipping
  - Overfitting        → Dropout, L2, EarlyStopping
BatchNorm:              después de Dense, antes de activación. Nunca antes de la salida
Optimizador:            Adam(lr=0.001) es el default recomendado
```

## S24 – CNN y Transfer Learning
```
Convolución:    filtro k×k desliza por la imagen → detecta patrones locales
padding='same': mantiene tamaño H×W   |   padding='valid': lo reduce
MaxPooling:     reduce H y W a la mitad
Flatten:        OBLIGATORIO entre Conv y Dense
input_shape:    siempre (H, W, C) con el canal
Normalizar:     imagen / 255.0

Transfer Learning:
  Fase 1: base.trainable = False → entrenar solo el top
  Fase 2: base.trainable = True, lr muy bajo → fine-tuning
  Siempre usar preprocess_input() del modelo correcto
  Data augmentation: SOLO en train, nunca en test
```

## S25 – Modelos Secuenciales
```
SimpleRNN: para secuencias cortas (<30 pasos)
LSTM:      para secuencias largas, tiene forget/input/output gate
GRU:       como LSTM pero más ligero

return_sequences:
  True  → si el siguiente es otro LSTM/GRU
  False → si el siguiente es Dense (o es el último LSTM)

Bidireccional: ✅ clasificación de texto | ❌ generación de texto | ❌ series de tiempo
Series de tiempo: reshape X → (N, pasos, 1) antes de LSTM
Padding: usar 'pre' para LSTM (los ceros al inicio, el texto real al final)
```

## S26 – Semántica Vectorial
```
One-hot:    dimensión = vocab_size, sin semántica, inviable para vocab grande
Embedding:  dimensión fija (64-768), denso, captura semántica

Word2Vec:   aprende embeddings por contexto. Una sola representación por palabra.
BERT:       embeddings CONTEXTUALES (misma palabra, vector diferente según contexto)
            → usar token [CLS] (posición 0) como representación de la oración

Similitud coseno = A·B / (|A||B|)   → 1=idénticos, 0=sin relación, -1=opuestos
PCA / t-SNE: reducir dimensión para visualizar embeddings
TF-IDF: alternativa clásica sin redes, no captura semántica
```

## S27 – Dilemas Éticos
```
Tipos de sesgo: representación, medición, agregación, histórico, retroalimentación
Fairness:       paridad demográfica / igualdad de oportunidad / calibración
               (imposible satisfacerlas todas simultáneamente)
Transparencia:  modelos interpretables vs caja negra → LIME, SHAP para explicar
Accountability: ¿quién responde cuando el modelo causa daño?
Checklist:      representación del dataset, variables proxy, costo asimétrico de errores
```